# Track 1 V5 | Synthetic residual-evidence submission
### A different signal, not another posterior-only boundary sweep

**Current best supplied by you:** Combined **1.4350**, Natural F1 **0.4349**, Synthetic F1 **0.8510**.
**Requested targets:** Natural F1 0.70 and/or Synthetic F1 0.95. These are objectives, not predicted results.

V5 prioritizes **Synthetic** and preserves every **Natural-routed** test row in the exact best JSONL. It does not train another ASR encoder, rerun V4, or reconstruct a different baseline.

The hypothesis: a library of noise waveforms recovered from *released BANK-only* noisy/clean pairs may identify repeated inserted sounds in other recordings. The older Track-2 experiment had promising denoising results **conditioned on known event windows**. That did not establish full-clip detection. This notebook tests full-clip detection instead.

**Pipeline:** BANK-only noisy minus clean -> waveform templates -> full noisy-clip FFT search -> sample-level verification -> confidence filtering -> conservative edits to Synthetic-routed predictions.

### Essential limits
- The method depends on recognizable noise waveforms recurring across clips. It can fail on unseen, differently cropped, stretched, or transformed noise.
- Query inference takes **only noisy audio** and the frozen BANK. It never receives a query's clean reference, GT windows, categories, or synthetic truth label.
- BANK, SELECT, and HOLD are source-prefix-disjoint for this run. The released validation set has been inspected in earlier experiments; **HOLD is not a fresh external test set**.
- Without an exact validation-prediction JSONL for the submitted 1.4350 stack, we cannot claim a paired improvement over it. The notebook labels any resulting ZIP **EXPERIMENTAL** and tests standalone matcher precision/false alarms instead. It never substitutes the old stable decoder and calls that your best baseline.
- No success threshold is relaxed after seeing HOLD. No new ZIP is written when the evidence gate fails; your original file remains untouched.

In [11]:
# 0. Configuration. No pip upgrades, no dependence on previous notebook globals.
from pathlib import Path
import os, sys
try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists(): drive.mount('/content/drive')
except ImportError:
    print('Not in Colab: edit WORK and data paths for this machine.')

WORK = Path('/content/drive/MyDrive/Datathon/work')
RUN_DIR = WORK / 'v5_synthetic_residual_evidence'
RUN_DIR.mkdir(parents=True, exist_ok=True)

# Exact candidate submitted most recently in this conversation. No silent fallback.
BEST_TEST_JSON = WORK / 'predictions_TODAY_6_V4_FORCE_dualunary.jsonl'
VAL_CACHE = WORK / 'official_val_posteriors_epoch_2.pkl'
TEST_POSTERIOR_CACHE = WORK / 'test_posteriors_wavlm_beats_epoch2.pkl'

GATE_JSON = WORK / 'submission_stage7_wavlm_indicw2v_fusion_diag.jsonl'
if not GATE_JSON.is_file():
    GATE_JSON = WORK / 'submission_stage7_wavlm_indicw2v_fusion_diag(1).jsonl'
GATE_THRESHOLD = 0.425
EXPECTED_TEST_CLIPS = 5517

VAL_ROOT = None   # Set an exact Path here to override the known locations below.
VAL_ROOT_CANDIDATES = [
    Path('/content/drive/MyDrive/Datathon/validation'),
    Path('/content/drive/MyDrive/PHD/Datathon/validation'),
    Path('/content/drive/MyDrive/validation'),
]
TEST_AUDIO_SOURCE = None  # An exact WAV folder or the input_data ZIP, when needed.
TEST_AUDIO_CANDIDATES = [
    Path('/content/data'),
    Path('/content/indoml_test_local'),
    Path('/content/test_wavs'),
    Path('/content/drive/MyDrive/Datathon/input_data.zip'),
    Path('/content/drive/MyDrive/PHD/Datathon/input_data.zip'),
    Path('/content/drive/MyDrive/PHD/Datathon'),
]

# Optional, ONLY if these are predictions of the SAME baseline stack on validation.
# An old stable-model proxy is NOT an exact baseline for the 1.4350 submission.
BASE_VAL_JSON = None
VAL_ROUTER_JSON = None
EXACT_VAL_PROVENANCE_CONFIRMED = False

# If no exact validation baseline exists, a precision-qualified experimental ZIP
# can be built. Its improvement over 1.4350 is NOT measured locally.
BUILD_EXPERIMENTAL_CANDIDATE = True
GENERATE_TEST_CANDIDATE = True
ALLOW_HIGH_CONFIDENCE_ADDITIONS = True

# Policy set BEFORE SELECT/HOLD. Leave unchanged after viewing HOLD results.
THRESHOLDS = [0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95]
SELECT_MIN_TP = 40
SELECT_MIN_PRECISION = 0.98
SELECT_MAX_NAT_FP_PER_CLIP = 0.01
HOLD_MIN_TP = 25
HOLD_MIN_PRECISION = 0.97
HOLD_PRECISION_WILSON_FLOOR = 0.90
HOLD_MAX_NAT_FP_PER_CLIP = 0.02
MAX_BOUNDARY_MOVE_SECONDS = 0.25
REPLACE_MIN_IOU = 0.50
MIN_ADD_QUALITY = 0.90

print('V5 SYNTHETIC RESIDUAL EVIDENCE -- no V6/V7 runtime needed')
print('Pinned best JSON:', BEST_TEST_JSON)
print('Output directory:', RUN_DIR)

V5 SYNTHETIC RESIDUAL EVIDENCE -- no V6/V7 runtime needed
Pinned best JSON: /content/drive/MyDrive/Datathon/work/predictions_TODAY_6_V4_FORCE_dualunary.jsonl
Output directory: /content/drive/MyDrive/Datathon/work/v5_synthetic_residual_evidence


## 1. Implementation and numerical tests
The implementation is embedded below: the notebook does not need a second Python file.
It uses NumPy, PyTorch, soundfile and pandas. FFTs run in float32; timing is rechecked
at the original 16-kHz sample grid. All setup tests use artificial signals, **not** your evaluation data.

In [2]:
"""Full-clip residual-template detector. No query labels/clean audio in inference."""
from __future__ import annotations
import os, io, re, json, math, time, hashlib, pickle, zipfile, copy
from pathlib import Path
from dataclasses import dataclass, asdict
from collections import OrderedDict, Counter
import numpy as np
import torch
import torch.nn.functional as F
import soundfile as sf

VERSION = 'residual_evidence_v5_20260917_r1'

@dataclass(frozen=True)
class MatcherConfig:
    sample_rate: int = 16000
    coarse_rate: int = 4000
    coarse_floor: float = 0.25
    template_batch: int = 32
    peaks_per_template: int = 3
    verify_topk: int = 128
    raw_radius: int = 12
    min_template_sec: float = 0.08
    max_template_sec: float = 6.0
    max_templates: int = 2000
    chunk_sec: float = 24.0
    fft_cache_mb: int = 384
    merge_iou: float = 0.65
    seed: int = 517


def digest(x):
    return hashlib.sha256(json.dumps(x, sort_keys=True, separators=(',', ':'), allow_nan=False).encode()).hexdigest()


def file_sha(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(4 << 20), b''):
            h.update(b)
    return h.hexdigest()


def wave_sha(y):
    return hashlib.sha256(np.asarray(y, '<f4').tobytes()).hexdigest()


def atomic_json(path, obj):
    p = Path(path); p.parent.mkdir(parents=True, exist_ok=True)
    tmp = p.with_name(p.name + '.tmp')
    with open(tmp, 'w', encoding='utf-8') as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, allow_nan=False)
    os.replace(tmp, p)


def atomic_pickle(path, obj):
    p = Path(path); p.parent.mkdir(parents=True, exist_ok=True)
    tmp = p.with_name(p.name + '.tmp')
    with open(tmp, 'wb') as f:
        pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(tmp, p)


def event_list(v):
    if isinstance(v, dict):
        if 'events' not in v:
            raise ValueError('Prediction row has no events field.')
        v = v['events']
    out = []
    for e in v:
        if isinstance(e, dict):
            a = float(e['onset'] if 'onset' in e else e['start'])
            b = float(e['offset'] if 'offset' in e else e['end'])
        else:
            a, b = float(e[0]), float(e[1])
        if not (math.isfinite(a) and math.isfinite(b) and 0 <= a < b):
            raise ValueError(f'Invalid event: {e}')
        out.append((a, b))
    return out


def read_jsonl(path, predictions=True):
    out = {}
    with open(path, encoding='utf-8') as f:
        for n, line in enumerate(f, 1):
            if not line.strip(): continue
            r = json.loads(line)
            cid = str(r['clip_id'])
            if cid in out: raise ValueError(f'Duplicate clip_id on line {n}: {cid}')
            if predictions: event_list(r)
            out[cid] = r
    if not out: raise ValueError(f'Empty JSONL: {path}')
    return out


def source_group(cid):
    # Best-effort recording prefix, NOT a claim of verified speaker identity.
    # Example: ...844425030213767_010_LowerDibangvalley-SPECIFIC_...
    m = re.search(r'_\d+_(?:GENERIC|[^_]+-SPECIFIC)_', str(cid))
    return str(cid)[:m.start()] if m else str(cid).rsplit('_', 3)[0]


def split_bucket(group):
    return int(hashlib.sha256(('v5_bank_source_' + group).encode()).hexdigest()[:12], 16) % 10


def bandlimited_resample(y, src, dst):
    """Windowed-sinc resampling, no SciPy/librosa dependency; common grid for pairs."""
    y = np.asarray(y, np.float32)
    if src == dst: return y.copy()
    n = int(round(len(y) * dst / src))
    if n < 1: raise ValueError('Audio became empty during resampling.')
    cutoff = min(1.0, float(dst) / src) * 0.94
    radius = int(math.ceil(20 / cutoff))
    offsets = np.arange(-radius, radius + 1)
    out = np.empty(n, np.float32)
    for a in range(0, n, 2048):
        b = min(n, a + 2048)
        pos = np.arange(a, b, dtype=np.float64) * src / dst
        ix = np.floor(pos).astype(np.int64)[:, None] + offsets[None]
        distance = pos[:, None] - ix
        w = cutoff * np.sinc(cutoff * distance)
        w *= np.where(np.abs(distance) < radius, 0.5 + 0.5*np.cos(np.pi*distance/radius), 0)
        valid = (ix >= 0) & (ix < len(y))
        w *= valid
        w /= np.maximum(w.sum(1, keepdims=True), 1e-12)
        out[a:b] = np.sum(y[np.clip(ix, 0, len(y)-1)] * w, axis=1)
    return out


def load_audio(path_or_bytes, sr=16000):
    obj = io.BytesIO(path_or_bytes) if isinstance(path_or_bytes, (bytes, bytearray)) else path_or_bytes
    y, fs = sf.read(obj, dtype='float32', always_2d=True)
    y = np.mean(y, axis=1, dtype=np.float32)
    if not len(y) or not np.isfinite(y).all(): raise ValueError(f'Invalid audio: {str(path_or_bytes)[:100]}')
    return bandlimited_resample(y, int(fs), sr)


def condition_wave(y):
    """Mild pre-emphasis, identically applied to templates and query recordings."""
    y = np.asarray(y, np.float32)
    return np.concatenate([y[:1], y[1:] - 0.85*y[:-1]]).astype(np.float32)


def coarse_wave(y, cfg):
    ratio = cfg.sample_rate // cfg.coarse_rate
    if ratio * cfg.coarse_rate != cfg.sample_rate:
        raise ValueError('coarse_rate must divide sample_rate.')
    y = condition_wave(y)
    t = np.arange(-32, 33, dtype=np.float64)
    cutoff = 0.90 / ratio
    kernel = cutoff * np.sinc(cutoff*t) * np.hanning(len(t))
    kernel /= kernel.sum()
    # Odd, symmetric FIR; zero padding preserves the time origin.
    padded = np.pad(y, (32, 32))
    return np.convolve(padded, kernel, mode='valid')[::ratio].astype(np.float32)


def iou(a, b):
    inter = max(0., min(a[1], b[1]) - max(a[0], b[0]))
    union = max(a[1], b[1]) - min(a[0], b[0])
    return inter / union if union > 0 else 0.


def closest_first_pairs(pred, ref):
    # Same explicitly named local scorer used in the user's notebooks.
    edges = []
    for i, (a,b) in enumerate(pred):
        for j, (s,e) in enumerate(ref):
            tol = max(0.20 * (e-s), 0.05)
            if abs(a-s) <= tol and abs(b-e) <= tol:
                edges.append((abs(a-s) + abs(b-e), i, j))
    used_p, used_r, pairs = set(), set(), []
    for _, i, j in sorted(edges):
        if i not in used_p and j not in used_r:
            used_p.add(i); used_r.add(j); pairs.append((i,j))
    return pairs


def score_events(pred, refs, durs, ids):
    tp = fp = fn = 0; dice = []; npred = ngt = 0
    for cid in ids:
        p, g = event_list(pred[cid]), event_list(refs[cid])
        nmatch = len(closest_first_pairs(p, g))
        tp += nmatch; fp += len(p)-nmatch; fn += len(g)-nmatch
        npred += len(p); ngt += len(g)
        n = max(1, int(round(durs[cid]/0.01)))
        pm = np.zeros(n, bool); gm = np.zeros(n, bool)
        for events, mask in ((p,pm),(g,gm)):
            for a,b in events:
                lo,hi = max(0,int(round(a/0.01))), min(n,int(round(b/0.01)))
                if hi > lo: mask[lo:hi] = True
        den = int(pm.sum()+gm.sum())
        dice.append(1. if den == 0 else 2.*np.logical_and(pm,gm).sum()/den)
    precision = tp/(tp+fp) if tp+fp else 0.
    recall = tp/(tp+fn) if tp+fn else 0.
    f1 = 2.*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.
    d = float(np.mean(dice)) if dice else 0.
    return dict(f1=f1, precision=precision, recall=recall, dice=d, combined=f1+d,
                tp=tp, fp=fp, fn=fn, clips=len(ids), predicted_events=npred, gt_events=ngt)


def wilson_lower(success, total, z=1.96):
    if total == 0: return 0.
    p = success/total; den = 1.+z*z/total
    return (p+z*z/(2*total)-z*math.sqrt(p*(1-p)/total+z*z/(4*total*total)))/den


def build_bank(bank_ids, refs, audio_paths, clean_paths, groups, cfg, progress=lambda x, **k: x):
    records, audits, source_hashes = [], [], []
    sr = cfg.sample_rate
    for cid in progress(bank_ids, desc='BANK pairs only: noisy minus clean'):
        x = load_audio(audio_paths[cid], sr)
        s = load_audio(clean_paths[cid], sr)
        if len(x) != len(s):
            raise ValueError(f'Pair length mismatch: {cid}; no silent truncation allowed.')
        mask = np.zeros(len(x), bool)
        for a,b in refs[cid]:
            mask[max(0,int(round(a*sr))):min(len(x),int(round(b*sr)))] = True
        outside = ~mask
        # Gain alignment is estimated from BANK-only non-event samples.
        scale = float(np.dot(x[outside],s[outside])/(np.dot(s[outside],s[outside])+1e-12)) if outside.sum()>sr//10 else 1.
        scale = float(np.clip(scale,0.8,1.2))
        residual = x - scale*s
        inside_rms = float(np.sqrt(np.mean(residual[mask]**2)+1e-20)) if mask.any() else 0.
        outside_rms = float(np.sqrt(np.mean(residual[outside]**2)+1e-20)) if outside.any() else 0.
        ratio = outside_rms/max(inside_rms,1e-10)
        good = bool(inside_rms>1e-5 and ratio<=0.15)
        audits.append(dict(cid=cid, scale=scale, inside_rms=inside_rms, outside_rms=outside_rms,
                           outside_inside_ratio=ratio, used=good))
        source_hashes.append(wave_sha(x))
        if not good: continue
        for j,(a,b) in enumerate(refs[cid]):
            # Mixed overlapping events are not single-template examples.
            if any(k!=j and min(b,e)-max(a,s0)>0.005 for k,(s0,e) in enumerate(refs[cid])): continue
            if not cfg.min_template_sec <= b-a <= cfg.max_template_sec: continue
            lo,hi = int(round(a*sr)), int(round(b*sr))
            if lo < 0 or hi > len(x) + int(.02*sr): raise ValueError(f'BANK label out of audio range: {cid}')
            hi = min(hi, len(x))
            w = residual[lo:hi].astype(np.float32)
            if len(w)<16 or float(np.std(w))<1e-5: continue
            coarse = coarse_wave(w,cfg)
            coarse -= coarse.mean()
            norm = float(np.linalg.norm(coarse))
            if norm < 1e-7: continue
            raw = condition_wave(w)
            # Reject templates with effectively just one energetic quarter.
            chunks = np.array([np.sum(v*v) for v in np.array_split(raw,4)])
            if np.sum(chunks > .02*max(float(chunks.sum()),1e-10)) < 2: continue
            records.append(dict(tid=f'{cid}::{j}', source_cid=cid, group=groups[cid],
                                wave=w, raw=raw, coarse=(coarse/norm).astype(np.float32),
                                annotation_duration=b-a))
    # Exact waveform deduplication only; no fitting on SELECT/HOLD.
    unique = {}; dedup = 0
    for r in records:
        w = r['wave']/max(float(np.max(np.abs(r['wave']))),1e-9)
        key = hashlib.sha256(np.round(w*32700).astype('<i2').tobytes()).hexdigest()
        if key in unique: dedup += 1
        else: unique[key] = r
    records = sorted(unique.values(), key=lambda r: hashlib.sha256(r['tid'].encode()).hexdigest())
    capped = max(0,len(records)-cfg.max_templates)
    records = records[:cfg.max_templates]
    if not records: raise RuntimeError('No valid additive-noise templates. Stop: the waveform-template hypothesis is unsupported for these pairs.')
    return dict(version=VERSION, cfg=asdict(cfg), records=records, audits=audits,
                bank_ids=list(bank_ids), bank_groups=sorted({groups[c] for c in bank_ids}),
                source_hashes=source_hashes, deduplicated=dedup, capped=capped)


class ResidualMatcher:
    def __init__(self, bank, cfg, device=None):
        self.cfg, self.bank = cfg, bank
        self.device = torch.device(device or ('cuda' if torch.cuda.is_available() else 'cpu'))
        self.records = bank['records']
        self.source_hashes = set(bank.get('source_hashes',[]))
        self.cache = OrderedDict(); self.cache_bytes = 0
        self.signature = digest(dict(version=VERSION,cfg=asdict(cfg),
            templates=[(r['tid'], wave_sha(r['wave'])) for r in self.records]))

    def _spectrum(self, ix, nfft):
        key = (tuple(ix), nfft)
        if key in self.cache:
            z = self.cache.pop(key); self.cache[key] = z
            return z
        t = torch.zeros((len(ix),nfft),dtype=torch.float32,device=self.device)
        for j,i in enumerate(ix):
            x = self.records[i]['coarse']; t[j,:len(x)] = torch.as_tensor(x,device=self.device)
        z = torch.fft.rfft(t,n=nfft,dim=-1)
        cost = z.numel()*z.element_size(); max_bytes = self.cfg.fft_cache_mb*(1<<20)
        while self.cache and self.cache_bytes+cost > max_bytes:
            _,old = self.cache.popitem(last=False); self.cache_bytes-=old.numel()*old.element_size()
        if cost<=max_bytes: self.cache[key] = z; self.cache_bytes+=cost
        return z

    @torch.inference_mode()
    def _coarse_scan(self,y,exclude_group=None):
        x = coarse_wave(y,self.cfg); n = len(x)
        eligible = [i for i,r in enumerate(self.records)
                    if len(r['coarse'])<=n and (exclude_group is None or r['group']!=exclude_group)]
        if not eligible: return []
        mmax = max(len(self.records[i]['coarse']) for i in eligible)
        nfft = 1<<(n+mmax-2).bit_length()
        xt = torch.as_tensor(x,dtype=torch.float32,device=self.device)
        xf = torch.fft.rfft(xt,n=nfft)
        # Use float64 prefix sums to avoid variance cancellation near silence.
        xx = xt.double()
        pref = torch.cat([torch.zeros(1,dtype=torch.float64,device=self.device),xx.cumsum(0)])
        pref2 = torch.cat([torch.zeros(1,dtype=torch.float64,device=self.device),(xx*xx).cumsum(0)])
        starts = torch.arange(n,device=self.device)[None]
        rows = []
        for a in range(0,len(eligible),self.cfg.template_batch):
            ix = eligible[a:a+self.cfg.template_batch]
            lens = torch.tensor([len(self.records[i]['coarse']) for i in ix],device=self.device)[:,None]
            tf = self._spectrum(ix,nfft)
            dot = torch.fft.irfft(xf[None]*tf.conj(),n=nfft,dim=-1)[:,:n]
            ends = (starts+lens).clamp(max=n)
            total = pref[ends]-pref[starts]; total2 = pref2[ends]-pref2[starts]
            energy = (total2-total*total/lens).clamp(min=1e-16).float()
            q = (dot.abs()/torch.sqrt(energy)).clamp(max=1.)
            q = q.masked_fill((starts+lens>n)|(energy<1e-12),-1.)
            for _ in range(self.cfg.peaks_per_template):
                vals, pos = q.max(dim=1)
                for j,(v,p) in enumerate(zip(vals.cpu().tolist(),pos.cpu().tolist())):
                    if v>=self.cfg.coarse_floor:
                        rows.append(dict(template_index=ix[j], sample4k=int(p), coarse=float(v)))
                rad = torch.maximum(lens//2,torch.full_like(lens,int(.06*self.cfg.coarse_rate)))
                q = q.masked_fill((starts-pos[:,None]).abs()<=rad,-1.)
        return sorted(rows,key=lambda r:r['coarse'],reverse=True)[:self.cfg.verify_topk]

    def _verify(self,raw_query,proposal):
        r = self.records[proposal['template_index']]; t = r['raw']; m = len(t)
        center = proposal['sample4k']*(self.cfg.sample_rate//self.cfg.coarse_rate)
        lo,hi = max(0,center-self.cfg.raw_radius), min(len(raw_query)-m,center+self.cfg.raw_radius)
        if hi<lo: return None
        t0=t.astype(np.float64)-np.mean(t,dtype=np.float64); ten=float(t0@t0)
        if ten<1e-12: return None
        seg=raw_query[lo:hi+m].astype(np.float64)
        dots=np.correlate(seg,t0,mode='valid')
        cs=np.r_[0.,np.cumsum(seg)]; cs2=np.r_[0.,np.cumsum(seg*seg)]
        sums=cs[m:]-cs[:-m]; energies=np.maximum(cs2[m:]-cs2[:-m]-sums*sums/m,1e-15)
        corr=dots/np.sqrt(energies*ten)
        j=int(np.argmax(np.abs(corr))); start=lo+j; sign=1. if corr[j]>=0 else -1.
        qseg=raw_query[start:start+m]
        chunks=[]
        for inds in np.array_split(np.arange(m),4):
            tt=t[inds].astype(np.float64); yy=qseg[inds].astype(np.float64)
            tt-=tt.mean(); yy-=yy.mean(); et=float(tt@tt)
            if et < .015*ten: continue
            chunks.append(float(sign*np.dot(tt,yy)/math.sqrt(max(et*float(yy@yy),1e-20))))
        if len(chunks)<2: return None
        raw_corr=float(abs(corr[j])); chunk=float(np.median(chunks))
        quality=float(np.clip(min(raw_corr,chunk),0.,1.))
        return dict(onset=start/self.cfg.sample_rate,offset=(start+m)/self.cfg.sample_rate,
                    quality=quality,corr16=raw_corr,corr4k=proposal['coarse'],chunk_median=chunk,
                    template_id=r['tid'],template_source=r['source_cid'],polarity=int(sign))

    def scan(self,y,exclude_group=None):
        """Only noisy waveform and a training bank are inputs. No event/GT arguments."""
        y=np.asarray(y,np.float32)
        if not np.isfinite(y).all(): raise ValueError('Nonfinite query audio.')
        if wave_sha(y) in self.source_hashes:
            raise RuntimeError('Query waveform exactly duplicates a BANK source. Refusing self-retrieval.')
        chunk=int(self.cfg.chunk_sec*self.cfg.sample_rate)
        overlap=int((self.cfg.max_template_sec+.05)*self.cfg.sample_rate)
        if chunk<=overlap: raise ValueError('chunk_sec must exceed max_template_sec.')
        starts=list(range(0,max(1,len(y)-overlap),chunk-overlap))
        found=[]
        for a in starts:
            yy=y[a:min(len(y),a+chunk)]
            if len(yy)<int(self.cfg.min_template_sec*self.cfg.sample_rate): continue
            raw=condition_wave(yy)
            for p in self._coarse_scan(yy,exclude_group):
                v=self._verify(raw,p)
                if v is not None:
                    v['onset']+=a/self.cfg.sample_rate; v['offset']+=a/self.cfg.sample_rate
                    found.append(v)
        # Keep geometric alternatives; final threshold/NMS is applied later.
        best={}
        for r in found:
            k=(round(r['onset'],4),round(r['offset'],4),r['template_id'])
            if k not in best or r['quality']>best[k]['quality']: best[k]=r
        return sorted(best.values(),key=lambda r:r['quality'],reverse=True)


def select_matches(rows, threshold, duplicate_iou=.65):
    chosen=[]
    for r in sorted(rows,key=lambda z:z['quality'],reverse=True):
        if r['quality']<threshold: continue
        ev=(r['onset'],r['offset'])
        if any(iou(ev,(x['onset'],x['offset']))>=duplicate_iou or
               (abs(ev[0]-x['onset'])<=.02 and abs(ev[1]-x['offset'])<=.02) for x in chosen): continue
        chosen.append(r)
    return sorted(chosen,key=lambda z:(z['onset'],z['offset']))


def patch_events(base, matches, max_move=.25, minimum_iou=.50, allow_add=False, add_threshold=.85):
    """Conservative edit: only mutual one-to-one overlaps; unmatched base retained."""
    base=event_list(base); proposal=[(m['onset'],m['offset']) for m in matches]
    edges=[]
    p2b={j:[] for j in range(len(proposal))}; b2p={i:[] for i in range(len(base))}
    for j,p in enumerate(proposal):
        for i,b in enumerate(base):
            ov=iou(p,b)
            if ov>=minimum_iou and max(abs(p[0]-b[0]),abs(p[1]-b[1]))<=max_move:
                p2b[j].append(i); b2p[i].append(j); edges.append((i,j))
    out=list(base); used=set(); changed=0
    for i,j in edges:
        if len(p2b[j])==len(b2p[i])==1:
            out[i]=proposal[j]; used.add(j); changed+=int(out[i]!=base[i])
    added=0
    if allow_add:
        for j,p in enumerate(proposal):
            if j in used or matches[j]['quality']<add_threshold: continue
            # No addition overlapping or touching an existing interval.
            if any(min(p[1],b[1]) - max(p[0],b[0]) > -.04 for b in out): continue
            out.append(p); added+=1
    return sorted(out),dict(changed=changed,added=added)


class WaveSource:
    """Read WAVs from one explicit folder/ZIP without extracting or guessing IDs."""
    def __init__(self,path,required_ids):
        self.path=str(path); self.archive=None; self.index={}
        if Path(path).is_file() and zipfile.is_zipfile(path):
            self.archive=zipfile.ZipFile(path)
            names=[i.filename for i in self.archive.infolist() if not i.is_dir() and i.filename.lower().endswith('.wav') and '__MACOSX' not in i.filename]
        elif Path(path).is_dir():
            names=[str(p) for p in Path(path).rglob('*.wav')]
        else: raise FileNotFoundError(path)
        for name in names:
            cid=Path(name).stem
            if cid not in required_ids: continue
            if cid in self.index: raise ValueError(f'Duplicate waveform stem in {path}: {cid}')
            self.index[cid]=name
        missing=set(required_ids)-set(self.index)
        if missing: raise ValueError(f'{path}: missing {len(missing)} required WAVs; examples {sorted(missing)[:3]}')
    def load(self,cid,sr=16000):
        loc=self.index[cid]
        return load_audio(self.archive.read(loc) if self.archive is not None else loc,sr)
    def duration(self,cid):
        loc=self.index[cid]
        x=io.BytesIO(self.archive.read(loc)) if self.archive is not None else loc
        info=sf.info(x)
        return info.frames/info.samplerate


def persist_query(matcher,y,path,exclude_group=None):
    wh=wave_sha(y); path=Path(path)
    signature=digest(dict(matcher=matcher.signature,wave=wh,exclude_group=exclude_group))
    if path.is_file():
        with open(path) as f: old=json.load(f)
        if old.get('signature')==signature: return old['proposals']
    rows=matcher.scan(y,exclude_group)
    atomic_json(path,dict(signature=signature,query_wave_sha256=wh,proposals=rows))
    return rows


def unit_tests():
    cfg=MatcherConfig(coarse_floor=.15,template_batch=2,peaks_per_template=2,verify_topk=20,fft_cache_mb=16)
    rng=np.random.default_rng(601)
    sr=cfg.sample_rate
    t=(rng.normal(0,.15,int(.36*sr))*np.hanning(int(.36*sr))).astype(np.float32)
    source=rng.normal(0,.02,int(1.8*sr)).astype(np.float32)
    a=1923; source[a:a+len(t)]+=t
    coarse=coarse_wave(t,cfg); coarse-=coarse.mean(); coarse/=np.linalg.norm(coarse)
    bank=dict(records=[dict(tid='test_template',source_cid='bank',group='g0',wave=t,raw=condition_wave(t),coarse=coarse)],source_hashes=[])
    m=ResidualMatcher(bank,cfg,device='cpu')
    got=select_matches(m.scan(source),.85)
    assert got and abs(got[0]['onset']-a/sr)<=1/sr, got
    assert abs(got[0]['offset']-(a+len(t))/sr)<=1/sr
    neg=m.scan(rng.normal(0,.02,len(source)).astype(np.float32))
    assert not select_matches(neg,.85)
    # Confirm no self-retrieval.
    m.source_hashes.add(wave_sha(source))
    try: m.scan(source)
    except RuntimeError: pass
    else: raise AssertionError('Self-retrieval guard failed.')
    refs={'x':[(1.,1.5)]}; durs={'x':2.}
    assert score_events(refs,refs,durs,['x'])['f1']==1.
    assert score_events({'x':[]},refs,durs,['x'])['fn']==1
    row=dict(onset=1.01,offset=1.51,quality=.99)
    patched,diag=patch_events([(1.,1.5)],[row])
    assert len(patched)==1 and diag['changed']==1
    patched,diag=patch_events([(1.,1.5)],[dict(onset=0.,offset=.3,quality=.99)])
    assert patched==[(1.,1.5)] and diag['added']==0
    print('PASS: FFT timing, raw sample alignment, negative audio, self-retrieval guard, scorer, conservative edits.')
    return True

In [3]:
# 2. Environment checks and isolated arithmetic tests.
import pandas as pd
from tqdm.auto import tqdm
from IPython.display import display

print('Python:', sys.version.split()[0], '| NumPy:', np.__version__, '| Torch:', torch.__version__)
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
# No package reinstall/downgrade in this notebook.
torch.set_num_threads(min(4, os.cpu_count() or 1))
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.deterministic = True
assert unit_tests()
CFG = MatcherConfig()

Python: 3.13.15 | NumPy: 2.1.3 | Torch: 2.11.0+cu128
Device: Tesla T4
PASS: FFT timing, raw sample alignment, negative audio, self-retrieval guard, scorer, conservative edits.


In [4]:
# 3. Input identity, exact IDs, and read-only best-submission snapshot.
required = [BEST_TEST_JSON, VAL_CACHE, GATE_JSON]
missing = [str(p) for p in required if not p.is_file()]
if missing:
    raise FileNotFoundError('Missing required files (no older baseline substituted):\n' + '\n'.join(missing))

BASE_TEST = read_jsonl(BEST_TEST_JSON)
TEST_GATE = read_jsonl(GATE_JSON, predictions=False)
if len(BASE_TEST) != EXPECTED_TEST_CLIPS or set(BASE_TEST) != set(TEST_GATE):
    raise ValueError('Base/router coverage mismatch. Confirm the exact submitted JSON and its router.')
for cid,r in TEST_GATE.items():
    q = float(r['p_syn'])
    if not np.isfinite(q) or not 0 <= q <= 1: raise ValueError(f'Invalid p_syn for {cid}')
NAT_ROUTE_IDS = sorted(c for c in BASE_TEST if float(TEST_GATE[c]['p_syn']) < GATE_THRESHOLD)
SYN_ROUTE_IDS = sorted(set(BASE_TEST) - set(NAT_ROUTE_IDS))
BASE_SHA = file_sha(BEST_TEST_JSON); GATE_SHA = file_sha(GATE_JSON)
atomic_json(RUN_DIR/'pinned_input_identity.json', {
    'version':VERSION,'base_path':str(BEST_TEST_JSON),'base_sha256':BASE_SHA,
    'router_path':str(GATE_JSON),'router_sha256':GATE_SHA,'gate_threshold':GATE_THRESHOLD,
    'natural_routed':len(NAT_ROUTE_IDS),'synthetic_routed':len(SYN_ROUTE_IDS),
    'reported_score_not_recomputed':{'combined':1.4350,'natural_f1':.4349,'synthetic_f1':.8510},
})
print('Base SHA256:',BASE_SHA)
print('Natural-routed rows to preserve:',len(NAT_ROUTE_IDS))
print('Synthetic-routed rows eligible for evidence-based edits:',len(SYN_ROUTE_IDS))

# This pickle is the user's trusted local cache, not downloaded executable content.
with open(VAL_CACHE,'rb') as f: raw = pickle.load(f)
records = raw if isinstance(raw,list) else [dict(v,clip_id=k) for k,v in raw.items()]
REFS={}; DURS={}; DOMAIN={}
for r in records:
    cid=str(r['clip_id'])
    if cid in REFS: raise ValueError(f'Duplicate validation ID: {cid}')
    flag=r['synthetic']
    if not isinstance(flag,(bool,np.bool_)): raise TypeError(f'synthetic is not Boolean for {cid}')
    REFS[cid]=event_list(r['gt'])
    DURS[cid]=float(r['duration'])
    DOMAIN[cid]='synthetic' if flag else 'natural'
    if not np.isfinite(DURS[cid]) or DURS[cid]<=0: raise ValueError(f'Invalid duration: {cid}')
VAL_SYN=sorted(c for c in REFS if DOMAIN[c]=='synthetic')
VAL_NAT=sorted(c for c in REFS if DOMAIN[c]=='natural')
print('Released validation:',len(REFS),'| Natural:',len(VAL_NAT),'| Synthetic:',len(VAL_SYN))
del raw, records

# Choose a root by the FULL required folder schema, never by a broad filename glob.
roots=[Path(VAL_ROOT)] if VAL_ROOT is not None else VAL_ROOT_CANDIDATES
valid_roots=[p for p in roots if all((p/s).is_dir() for s in
             ['naturalNoisyAudio','syntheticNoiseAudio','syntheticCleanRefAudio'])]
if not valid_roots:
    raise FileNotFoundError('No complete released validation root. Set VAL_ROOT to its exact folder.')
VAL_ROOT=valid_roots[0]

def index_wavs(folder, ids):
    out={}
    for p in folder.rglob('*.wav'):
        cid=p.stem
        if cid not in ids: continue
        if cid in out: raise ValueError(f'Duplicate stem in {folder}: {cid}')
        out[cid]=str(p)
    miss=set(ids)-set(out)
    if miss: raise FileNotFoundError(f'{folder}: missing {len(miss)} WAVs, e.g. {sorted(miss)[:3]}')
    return out

AUDIO_PATHS={**index_wavs(VAL_ROOT/'naturalNoisyAudio',set(VAL_NAT)),
             **index_wavs(VAL_ROOT/'syntheticNoiseAudio',set(VAL_SYN))}
CLEAN_PATHS=index_wavs(VAL_ROOT/'syntheticCleanRefAudio',set(VAL_SYN))
print('Using validation root:',VAL_ROOT)

Base SHA256: 4ed93d85aab88dce8f8b90504f112f0235228829e5f1225eb01927e2c82ad773
Natural-routed rows to preserve: 2957
Synthetic-routed rows eligible for evidence-based edits: 2560
Released validation: 2501 | Natural: 1806 | Synthetic: 695
Using validation root: /content/drive/MyDrive/PHD/Datathon/validation


In [5]:
# 4. Freeze BANK/SELECT/HOLD membership before building any template.
GROUPS={c:source_group(c) for c in REFS}
BANK_IDS=[c for c in VAL_SYN if split_bucket(GROUPS[c])<=5]
SELECT_SYN=[c for c in VAL_SYN if split_bucket(GROUPS[c]) in (6,7)]
HOLD_SYN=[c for c in VAL_SYN if split_bucket(GROUPS[c]) in (8,9)]
SELECT_NAT=[c for c in VAL_NAT if split_bucket(GROUPS[c]) in (6,7)]
HOLD_NAT=[c for c in VAL_NAT if split_bucket(GROUPS[c]) in (8,9)]
for a,b in [(BANK_IDS,SELECT_SYN),(BANK_IDS,HOLD_SYN),(SELECT_SYN,HOLD_SYN)]:
    assert {GROUPS[c] for c in a}.isdisjoint({GROUPS[c] for c in b})
if min(len(BANK_IDS),len(SELECT_SYN),len(HOLD_SYN))<40:
    raise RuntimeError('Too few clips in a source-disjoint split; inspect group IDs rather than silently splitting by event.')
if min(len(SELECT_NAT),len(HOLD_NAT))<100:
    raise RuntimeError('Too few natural control clips for the registered false-alarm test.')
print('BANK / SELECT / HOLD synthetic:',len(BANK_IDS),len(SELECT_SYN),len(HOLD_SYN))
print('SELECT / HOLD natural controls:',len(SELECT_NAT),len(HOLD_NAT))
print('Distinct source prefixes:',len(set(GROUPS.values())))
print('Example group IDs:',[(c,GROUPS[c]) for c in VAL_SYN[:2]])
print('HOLD is excluded from V5 fitting/selection; it is NOT a previously unseen external dataset.')
SPLIT=dict(bank=BANK_IDS,select_syn=SELECT_SYN,hold_syn=HOLD_SYN,
           select_nat=SELECT_NAT,hold_nat=HOLD_NAT,group_key=GROUPS)
SPLIT_SHA=digest(SPLIT)
atomic_json(RUN_DIR/'frozen_split.json',SPLIT)

BANK / SELECT / HOLD synthetic: 426 128 141
SELECT / HOLD natural controls: 338 408
Distinct source prefixes: 1982
Example group IDs: [('IISc_VaaniProject_K_ArunachalPradesh_LowerDibangvalley_LowerDibangvalley844425030213767_010_LowerDibangvalley-SPECIFIC_01478_3234_6516', 'IISc_VaaniProject_K_ArunachalPradesh_LowerDibangvalley_LowerDibangvalley844425030213767'), ('IISc_VaaniProject_K_ArunachalPradesh_LowerDibangvalley_LowerDibangvalley844425030213767_018_GENERIC_0791_12150_15358', 'IISc_VaaniProject_K_ArunachalPradesh_LowerDibangvalley_LowerDibangvalley844425030213767')]
HOLD is excluded from V5 fitting/selection; it is NOT a previously unseen external dataset.


## 5. Recover the training noise library
Only BANK clean references are opened below. SELECT/HOLD clean references are not read.
A bank-pair audit checks whether subtraction produces localized noise rather than widespread
speech residue. Overlapping multi-event snippets are excluded from the single-template bank.
The frozen BANK is also used at test time; the library is **not enlarged after validation**.

In [6]:
# 5. Resume-safe BANK creation, keyed by data identity and configuration.
BANK_PATH=RUN_DIR/'frozen_bank.pkl'
BANK_INPUT_KEY=digest(dict(version=VERSION,config=asdict(CFG),split=SPLIT_SHA,
    val_cache_sha=file_sha(VAL_CACHE),root=str(VAL_ROOT),
    files=[(c,os.path.getsize(AUDIO_PATHS[c]),os.path.getmtime(AUDIO_PATHS[c]),
             os.path.getsize(CLEAN_PATHS[c]),os.path.getmtime(CLEAN_PATHS[c])) for c in BANK_IDS]))
BANK=None
if BANK_PATH.is_file():
    with open(BANK_PATH,'rb') as f: candidate=pickle.load(f)
    if candidate.get('input_key')==BANK_INPUT_KEY: BANK=candidate
if BANK is None:
    BANK=build_bank(BANK_IDS,REFS,AUDIO_PATHS,CLEAN_PATHS,GROUPS,CFG,progress=tqdm)
    BANK['input_key']=BANK_INPUT_KEY
    atomic_pickle(BANK_PATH,BANK)
assert set(BANK['bank_ids'])==set(BANK_IDS)
MATCHER=ResidualMatcher(BANK,CFG)
BANK_AUDIT=pd.DataFrame(BANK['audits'])
BANK_AUDIT.to_csv(RUN_DIR/'bank_pair_audit.csv',index=False)
print('Usable templates:',len(BANK['records']))
print('Bank pairs passing additive-noise audit:',int(BANK_AUDIT.used.sum()),'/',len(BANK_AUDIT))
print('Exact duplicates removed:',BANK['deduplicated'],'| templates omitted by preset cap:',BANK['capped'])
print('Median outside/inside residual ratio:',BANK_AUDIT.outside_inside_ratio.median())
print('Matcher signature:',MATCHER.signature)
if len(BANK['records'])<20:
    print('Small template library: coverage may be low; measured precision/recall, not library size, will decide.')

BANK pairs only: noisy minus clean:   0%|          | 0/426 [00:00<?, ?it/s]

Usable templates: 1349
Bank pairs passing additive-noise audit: 426 / 426
Exact duplicates removed: 0 | templates omitted by preset cap: 0
Median outside/inside residual ratio: 0.0007129968113285673
Matcher signature: ad80f13fc9d2f853ba6609906100c870757973829fa040c7c6e9eff0408dbf0a


In [7]:
# 6. Full-clip SELECT retrieval. Ground truth is not an input to MATCHER.scan.
# Only noisy audio is opened here. Per-clip caches make this safe to resume.

def retrieve_ids(ids, label):
    out={}; started=time.time()
    for j,cid in enumerate(tqdm(ids,desc=label)):
        y=load_audio(AUDIO_PATHS[cid],CFG.sample_rate)
        if abs(len(y)/CFG.sample_rate-DURS[cid])>.04:
            raise ValueError(f'Validation cache/audio duration mismatch for {cid}. Check the dataset version.')
        out[cid]=persist_query(MATCHER,y,RUN_DIR/'queries'/f'{cid}.json',exclude_group=GROUPS[cid])
        if j==9:
            print(f'First 10 queries: {(time.time()-started):.1f}s; remaining timing depends on audio length and cache hits.')
    return out

SELECT_PROPOSALS=retrieve_ids(SELECT_SYN+SELECT_NAT,'SELECT full-clip noise-template search')
print('SELECT query clips:',len(SELECT_PROPOSALS))

SELECT full-clip noise-template search:   0%|          | 0/466 [00:00<?, ?it/s]

First 10 queries: 6.5s; remaining timing depends on audio length and cache hits.
SELECT query clips: 466


## 7. Select a high-precision operating point
The raw waveform match score is **not a calibrated probability**. Thresholds are selected
using actual one-to-one event matches. Natural clips are controls for accidental template matches.

If no exact baseline validation predictions are provided, this measures standalone retrieval,
not the gain of the final hybrid. Boundary replacements retain unmatched base events; additions
need still stronger waveform evidence. No detector event is allowed to delete an uncertain
baseline event merely because a template was not found.

In [8]:
# 7. Optional exact validation baseline, plus SELECT-only threshold selection.
EXACT_BASELINE_MODE=bool(BASE_VAL_JSON is not None and VAL_ROUTER_JSON is not None and EXACT_VAL_PROVENANCE_CONFIRMED)
BASE_VAL=None; VAL_GATE=None
if EXACT_BASELINE_MODE:
    BASE_VAL=read_jsonl(Path(BASE_VAL_JSON))
    VAL_GATE=read_jsonl(Path(VAL_ROUTER_JSON),predictions=False)
    required=set(SELECT_SYN+SELECT_NAT+HOLD_SYN+HOLD_NAT)
    if not required<=set(BASE_VAL) or not required<=set(VAL_GATE):
        raise ValueError('Exact validation base/router do not cover evaluation IDs. Do not substitute another version.')
    if set(BASE_VAL)&set(BASE_TEST):
        raise ValueError('Validation and test IDs overlap; confirm the intended datasets.')
    print('PAIRED MODE: baseline provenance explicitly confirmed by user.')
else:
    print('UNPAIRED MODE: no exact validation output for the 1.4350 stack. All ZIPs are EXPERIMENTAL.')


def detection_map(rows_map, ids, threshold):
    return {c:[(r['onset'],r['offset']) for r in select_matches(rows_map[c],threshold,CFG.merge_iou)] for c in ids}


def hybrid_map(rows_map, ids, threshold):
    result={}
    for cid in ids:
        b=event_list(BASE_VAL[cid])
        if float(VAL_GATE[cid]['p_syn'])<GATE_THRESHOLD:
            result[cid]=b
        else:
            mm=select_matches(rows_map[cid],threshold,CFG.merge_iou)
            result[cid],_=patch_events(b,mm,MAX_BOUNDARY_MOVE_SECONDS,REPLACE_MIN_IOU,
                ALLOW_HIGH_CONFIDENCE_ADDITIONS,max(MIN_ADD_QUALITY,min(.99,threshold+.10)))
    return result

SELECT_ALL=SELECT_SYN+SELECT_NAT
SELECT_BASE_SCORE=score_events(BASE_VAL,REFS,DURS,SELECT_ALL) if EXACT_BASELINE_MODE else None
select_rows=[]
for threshold in THRESHOLDS:
    pred=detection_map(SELECT_PROPOSALS,SELECT_ALL,threshold)
    syn=score_events(pred,REFS,DURS,SELECT_SYN)
    nat=score_events(pred,REFS,DURS,SELECT_NAT)
    qualified=(syn['tp']>=SELECT_MIN_TP and syn['precision']>=SELECT_MIN_PRECISION
               and nat['fp']/len(SELECT_NAT)<=SELECT_MAX_NAT_FP_PER_CLIP)
    entry={'threshold':threshold,'synthetic_f1':syn['f1'],'synthetic_precision':syn['precision'],
        'synthetic_recall':syn['recall'],'synthetic_dice':syn['dice'],'syn_tp':syn['tp'],'syn_fp':syn['fp'],
        'natural_false_events_per_clip':nat['fp']/len(SELECT_NAT),'qualified':qualified}
    if EXACT_BASELINE_MODE:
        hybrid=score_events(hybrid_map(SELECT_PROPOSALS,SELECT_ALL,threshold),REFS,DURS,SELECT_ALL)
        entry['hybrid_combined_gain']=hybrid['combined']-SELECT_BASE_SCORE['combined']
        qualified=qualified and entry['hybrid_combined_gain']>0
        entry['qualified']=qualified
    select_rows.append(entry)
SELECT_TABLE=pd.DataFrame(select_rows)
display(SELECT_TABLE)
SELECT_TABLE.to_csv(RUN_DIR/'select_thresholds.csv',index=False)
eligible=SELECT_TABLE[SELECT_TABLE.qualified].copy()
SELECT_PASS=not eligible.empty
FROZEN=None
if SELECT_PASS:
    order=['hybrid_combined_gain','synthetic_f1','threshold'] if EXACT_BASELINE_MODE else ['synthetic_recall','synthetic_f1','threshold']
    chosen=eligible.sort_values(order,ascending=False).iloc[0]
    FROZEN={'threshold':float(chosen.threshold),'max_move':MAX_BOUNDARY_MOVE_SECONDS,'min_iou':REPLACE_MIN_IOU,
        'allow_add':ALLOW_HIGH_CONFIDENCE_ADDITIONS,
        'add_threshold':max(MIN_ADD_QUALITY,min(.99,float(chosen.threshold)+.10)),
        'matcher_signature':MATCHER.signature,'bank_input_key':BANK_INPUT_KEY,'split_sha':SPLIT_SHA,
        'base_sha':BASE_SHA,'exact_validation_baseline':EXACT_BASELINE_MODE,
        'hold_criteria':{'min_tp':HOLD_MIN_TP,'min_precision':HOLD_MIN_PRECISION,
                         'wilson_precision_floor':HOLD_PRECISION_WILSON_FLOOR,
                         'max_natural_fp_per_clip':HOLD_MAX_NAT_FP_PER_CLIP}}
    atomic_json(RUN_DIR/'frozen_before_hold.json',FROZEN)
    print('FROZEN BEFORE HOLD:',FROZEN)
else:
    print('SELECT FAIL: no high-precision operating point. HOLD and submission creation will be skipped.')
    print('This means the residual-template hypothesis did not qualify; keep 1.4350.')

UNPAIRED MODE: no exact validation output for the 1.4350 stack. All ZIPs are EXPERIMENTAL.


,threshold,synthetic_f1,synthetic_precision,synthetic_recall,synthetic_dice,syn_tp,syn_fp,natural_false_events_per_clip,qualified
0,0.40,0.907895,0.985714,0.841463,0.913432,345,5,0.017751,False
1,0.45,0.906456,0.985673,0.839024,0.910428,344,5,0.002959,True
2,0.50,0.904762,0.988439,0.834146,0.905394,342,4,0.000000,True
3,0.55,0.903054,0.991254,0.829268,0.900524,340,3,0.000000,True
4,0.60,0.898396,0.994083,0.819512,0.887812,336,2,0.000000,True
5,0.65,0.896921,0.994065,0.817073,0.884949,335,2,0.000000,True
6,0.70,0.889488,0.993976,0.804878,0.873897,330,2,0.000000,True
7,0.75,0.875853,0.993808,0.782927,0.850486,321,2,0.000000,True
8,0.80,0.865014,0.993671,0.765854,0.829767,314,2,0.000000,True
9,0.85,0.855556,0.993548,0.751220,0.811848,308,2,0.000000,True


FROZEN BEFORE HOLD: {'threshold': 0.45, 'max_move': 0.25, 'min_iou': 0.5, 'allow_add': True, 'add_threshold': 0.9, 'matcher_signature': 'ad80f13fc9d2f853ba6609906100c870757973829fa040c7c6e9eff0408dbf0a', 'bank_input_key': '06e03ffe8e16a8477ee160f4a2c4773f44180a0dec5b94f9303b1d8540b1debd', 'split_sha': 'e2a6f36cb70a33b472211bbc5137b807a593e894399db9aeb6a628c3c583cbb6', 'base_sha': '4ed93d85aab88dce8f8b90504f112f0235228829e5f1225eb01927e2c82ad773', 'exact_validation_baseline': False, 'hold_criteria': {'min_tp': 25, 'min_precision': 0.97, 'wilson_precision_floor': 0.9, 'max_natural_fp_per_clip': 0.02}}


In [9]:
# 8. HOLD evaluation, once, after freezing. No threshold retuning or bank expansion.
EVIDENCE_PASS=False; PAIRED_IMPROVEMENT=False; TARGET_095_OBSERVED=False
HOLD_REPORT={'select_pass':bool(SELECT_PASS),'version':VERSION,'historically_reused_validation':True}
if SELECT_PASS:
    HOLD_ALL=HOLD_SYN+HOLD_NAT
    HOLD_PROPOSALS=retrieve_ids(HOLD_ALL,'HOLD full-clip noise-template search')
    ph=detection_map(HOLD_PROPOSALS,HOLD_ALL,FROZEN['threshold'])
    syn=score_events(ph,REFS,DURS,HOLD_SYN)
    nat=score_events(ph,REFS,DURS,HOLD_NAT)
    precision_lower=wilson_lower(syn['tp'],syn['tp']+syn['fp'])
    EVIDENCE_PASS=bool(syn['tp']>=HOLD_MIN_TP and syn['precision']>=HOLD_MIN_PRECISION
        and precision_lower>=HOLD_PRECISION_WILSON_FLOOR
        and nat['fp']/len(HOLD_NAT)<=HOLD_MAX_NAT_FP_PER_CLIP)
    TARGET_095_OBSERVED=bool(syn['f1']>=.95)
    HOLD_REPORT.update({'synthetic_standalone':syn,'natural_control':nat,
        'precision_wilson_lower_95':precision_lower,'evidence_pass':EVIDENCE_PASS,
        'target_095_standalone_observed':TARGET_095_OBSERVED})
    display(pd.DataFrame([dict(domain='Synthetic matcher ONLY',**syn),dict(domain='Natural control matcher ONLY',**nat)]))
    print('Synthetic precision Wilson lower bound:',round(precision_lower,4))
    print('TARGET 0.95 observed on standalone synthetic HOLD:',TARGET_095_OBSERVED)
    print('Natural 0.70 not attempted: the Natural-routed submission rows are frozen.')
    if EXACT_BASELINE_MODE:
        hb=score_events(BASE_VAL,REFS,DURS,HOLD_ALL)
        hhpred=hybrid_map(HOLD_PROPOSALS,HOLD_ALL,FROZEN['threshold'])
        hh=score_events(hhpred,REFS,DURS,HOLD_ALL)
        bn=score_events(BASE_VAL,REFS,DURS,HOLD_NAT)
        hn=score_events(hhpred,REFS,DURS,HOLD_NAT)
        PAIRED_IMPROVEMENT=bool(hh['combined']>=hb['combined']+.003 and hh['f1']>hb['f1']
                               and hh['dice']>=hb['dice']-.003 and hn['combined']>=bn['combined']-.002)
        HOLD_REPORT.update({'exact_baseline':hb,'patched_hybrid':hh,'paired_improvement':PAIRED_IMPROVEMENT})
        display(pd.DataFrame([dict(system='Exact baseline',**hb),dict(system='V5 patched hybrid',**hh)]))
    else:
        print('No paired improvement over 1.4350 has been established; this is a precision-qualified experiment only.')

CAN_BUILD=bool(EVIDENCE_PASS and (PAIRED_IMPROVEMENT if EXACT_BASELINE_MODE else BUILD_EXPERIMENTAL_CANDIDATE))
HOLD_REPORT['can_build_experimental_candidate']=CAN_BUILD
atomic_json(RUN_DIR/'hold_report.json',HOLD_REPORT)
print('EVIDENCE_PASS:',EVIDENCE_PASS,'| PAIRED_IMPROVEMENT:',PAIRED_IMPROVEMENT,'| CAN_BUILD:',CAN_BUILD)
print('Original baseline and decision thresholds have not been changed.')

HOLD full-clip noise-template search:   0%|          | 0/549 [00:00<?, ?it/s]

First 10 queries: 4.4s; remaining timing depends on audio length and cache hits.


,domain,f1,precision,recall,dice,combined,tp,fp,fn,clips,predicted_events,gt_events
0,Synthetic matcher ONLY,0.903382,0.992042,0.829268,0.895845,1.799227,374,3,77,141,377,451
1,Natural control matcher ONLY,0.040876,0.823529,0.020958,0.022750,0.063626,14,3,654,408,17,668


Synthetic precision Wilson lower bound: 0.9769
TARGET 0.95 observed on standalone synthetic HOLD: False
Natural 0.70 not attempted: the Natural-routed submission rows are frozen.
No paired improvement over 1.4350 has been established; this is a precision-qualified experiment only.
EVIDENCE_PASS: True | PAIRED_IMPROVEMENT: False | CAN_BUILD: True
Original baseline and decision thresholds have not been changed.


In [12]:
# 9. Load test WAVs only AFTER the evidence gate; exact IDs checked up front.
TEST_SOURCE=None
if CAN_BUILD and GENERATE_TEST_CANDIDATE:
    options=[Path(TEST_AUDIO_SOURCE)] if TEST_AUDIO_SOURCE is not None else TEST_AUDIO_CANDIDATES
    errors=[]
    for path in options:
        if not path.exists(): continue
        try:
            TEST_SOURCE=WaveSource(path,set(BASE_TEST)); break
        except (ValueError,FileNotFoundError,zipfile.BadZipFile) as e:
            errors.append(f'{path}: {e}')
    if TEST_SOURCE is None:
        raise FileNotFoundError('No complete test audio source. Set TEST_AUDIO_SOURCE explicitly.\n'+'\n'.join(errors))
    print('Verified test audio source:',TEST_SOURCE.path)
    assert file_sha(BEST_TEST_JSON)==BASE_SHA and file_sha(GATE_JSON)==GATE_SHA
    if set(BASE_TEST)&set(REFS):
        raise RuntimeError('Validation/test ID overlap. Refusing to treat a labelled input as hidden test data.')
else:
    print('Test stage skipped: no evidence-qualified candidate to construct.')

Verified test audio source: /content/drive/MyDrive/PHD/Datathon


In [13]:
# 10. Produce one hybrid candidate. No new neural inference or baseline reconstruction.
FINAL=None; TEST_EDIT_LOG=[]; OUTPUT_ZIP=None
if CAN_BUILD and GENERATE_TEST_CANDIDATE:
    FINAL=copy.deepcopy(BASE_TEST)
    TEST_DURS={}
    # Optional cached durations avoid reopening all unchanged WAVs during the audit.
    if TEST_POSTERIOR_CACHE.is_file():
        with open(TEST_POSTERIOR_CACHE,'rb') as f: prior=pickle.load(f)
        if isinstance(prior,dict):
            for cid,z in prior.items():
                d=z.get('dur',z.get('duration'))
                if d is not None and np.isfinite(float(d)): TEST_DURS[str(cid)]=float(d)
        del prior
    started=time.time()
    for j,cid in enumerate(tqdm(SYN_ROUTE_IDS,desc='Test Synthetic-route waveform evidence')):
        y=TEST_SOURCE.load(cid,CFG.sample_rate)
        TEST_DURS[cid]=len(y)/CFG.sample_rate
        proposals=persist_query(MATCHER,y,RUN_DIR/'test_queries'/f'{cid}.json')
        matches=select_matches(proposals,FROZEN['threshold'],CFG.merge_iou)
        original=event_list(BASE_TEST[cid])
        edited,info=patch_events(original,matches,FROZEN['max_move'],FROZEN['min_iou'],
                                FROZEN['allow_add'],FROZEN['add_threshold'])
        if edited!=original:
            candidate=[{'onset':round(a,3),'offset':round(b,3)} for a,b in edited]
            # Validate after rounding too; rejected edits revert the WHOLE clip.
            if all(0<=e['onset']<e['offset']<=TEST_DURS[cid]+.001 for e in candidate):
                FINAL[cid]['events']=candidate
            else:
                info['reverted_invalid_rounding']=True
        TEST_EDIT_LOG.append(dict(clip_id=cid,accepted_matches=len(matches),
            clip_changed=FINAL[cid]['events']!=BASE_TEST[cid]['events'],
            old_count=len(original),new_count=len(FINAL[cid]['events']),**info))
        if j==9:
            print(f'First 10 test clips took {time.time()-started:.1f}s. Predictions are checkpointed clip by clip.')
    for cid in BASE_TEST:
        if cid not in TEST_DURS: TEST_DURS[cid]=TEST_SOURCE.duration(cid)
    pd.DataFrame(TEST_EDIT_LOG).to_csv(RUN_DIR/'test_edit_audit.csv',index=False)

Test Synthetic-route waveform evidence:   0%|          | 0/2560 [00:00<?, ?it/s]

First 10 test clips took 6.1s. Predictions are checkpointed clip by clip.


In [14]:
# 11. Strict final audit, then write a NEW ZIP. Original files are never overwritten.
if FINAL is not None:
    assert set(FINAL)==set(BASE_TEST)
    for cid in NAT_ROUTE_IDS:
        assert FINAL[cid]==BASE_TEST[cid],f'Natural-routed row changed: {cid}'
    for cid,r in FINAL.items():
        for a,b in event_list(r):
            if b>TEST_DURS[cid]+.05: raise ValueError(f'Output boundary exceeds audio duration: {cid}')
    changed=sum(FINAL[c]['events']!=BASE_TEST[c]['events'] for c in BASE_TEST)
    additions=sum(max(0,len(FINAL[c]['events'])-len(BASE_TEST[c]['events'])) for c in BASE_TEST)
    assert file_sha(BEST_TEST_JSON)==BASE_SHA, 'Input baseline changed during the run.'
    if changed==0:
        print('NO NEW SUBMISSION: every row equals the existing best. Do not spend a slot on a duplicate ZIP.')
    else:
        suffix='PAIRED_VALIDATED' if EXACT_BASELINE_MODE and PAIRED_IMPROVEMENT else 'EXPERIMENTAL'
        stem=f'submission_V5_SYNTH_RESIDUAL_{suffix}_{BASE_SHA[:8]}'
        output_json=RUN_DIR/(stem+'.jsonl')
        OUTPUT_ZIP=RUN_DIR/(stem+'.zip')
        tmp_json=output_json.with_suffix('.jsonl.tmp')
        with open(tmp_json,'w',encoding='utf-8') as f:
            for cid in sorted(FINAL):
                f.write(json.dumps(FINAL[cid],ensure_ascii=False,allow_nan=False)+'\n')
        os.replace(tmp_json,output_json)
        tmp_zip=OUTPUT_ZIP.with_suffix('.zip.tmp')
        with zipfile.ZipFile(tmp_zip,'w',zipfile.ZIP_DEFLATED) as z:
            z.write(output_json,arcname='predictions.jsonl')
        with zipfile.ZipFile(tmp_zip) as z:
            assert z.namelist()==['predictions.jsonl'] and z.testzip() is None
            check=[json.loads(line) for line in z.read('predictions.jsonl').decode().splitlines() if line.strip()]
            assert len(check)==EXPECTED_TEST_CLIPS and len({r['clip_id'] for r in check})==EXPECTED_TEST_CLIPS
        os.replace(tmp_zip,OUTPUT_ZIP)
        manifest=dict(version=VERSION,base_sha=BASE_SHA,output_sha=file_sha(output_json),
            matcher_sha=MATCHER.signature,split_sha=SPLIT_SHA,frozen_policy=FROZEN,
            validation_status=suffix,natural_routed_rows_unchanged=len(NAT_ROUTE_IDS),
            synthetic_routed_rows_considered=len(SYN_ROUTE_IDS),changed_clips=changed,added_events=additions,
            target_095_observed_on_standalone_hold=TARGET_095_OBSERVED,
            actual_leaderboard_score='UNKNOWN until organizer evaluation',zip_path=str(OUTPUT_ZIP))
        atomic_json(RUN_DIR/(stem+'_manifest.json'),manifest)
        print('='*76)
        print('V5 CANDIDATE READY --',suffix)
        print('Natural-routed rows exactly unchanged:',len(NAT_ROUTE_IDS))
        print('Synthetic-routed rows changed:',changed,'| added events:',additions)
        print('Bank used at test is the SAME frozen bank used on SELECT and HOLD.')
        print('The test F1 is unknown; 0.95/0.70 are NOT claimed.')
        print('ZIP:',OUTPUT_ZIP)
        print('MANIFEST:',RUN_DIR/(stem+'_manifest.json'))
else:
    print('No V5 ZIP generated. Preserve the 1.4350 submission; the experiment did not clear its preset gate.')

V5 CANDIDATE READY -- EXPERIMENTAL
Natural-routed rows exactly unchanged: 2957
Synthetic-routed rows changed: 1930 | added events: 714
Bank used at test is the SAME frozen bank used on SELECT and HOLD.
The test F1 is unknown; 0.95/0.70 are NOT claimed.
ZIP: /content/drive/MyDrive/Datathon/work/v5_synthetic_residual_evidence/submission_V5_SYNTH_RESIDUAL_EXPERIMENTAL_4ed93d85.zip
MANIFEST: /content/drive/MyDrive/Datathon/work/v5_synthetic_residual_evidence/submission_V5_SYNTH_RESIDUAL_EXPERIMENTAL_4ed93d85_manifest.json


## Read the result correctly

- `TARGET 0.95 observed ... True` refers to **measured standalone synthetic detection on this released HOLD subset**, not hidden test performance.
- `EXPERIMENTAL` means exact best-baseline validation outputs were unavailable. High matcher precision does not establish that every hybrid replacement improves the already-strong baseline.
- Unchanged Natural-*routed* rows do not imply that all true Natural clips are unchanged: the existing router can misroute some Natural clips. Natural validation controls measure that risk, but cannot eliminate it.
- High residual-template performance would support repeated-waveform recognition on this dataset. It would not establish general robustness to unseen natural noises.
- A failed evidence gate is a result. Do not lower the preset thresholds after looking at HOLD.

### Outputs to inspect
`SELECT threshold table`, `HOLD standalone matcher table`, `EVIDENCE_PASS`,
`TARGET 0.95 observed`, and the final `V5 CANDIDATE READY` block.

## Provenance and references
The design uses the user's existing Track-1 JSONL format and 10-ms, closest-first local scorer,
and the BANK-only residual idea from `IndoML_Track2_Rank1_TypeConditioned_Denoiser_Benchmark.ipynb`.
The full-clip matcher here is new; the old denoiser's GT-conditioned result is not substituted for it.

Official competition page: https://indoml.in/datathon/ (accessed 17 September 2026).
It lists released validation for tuning/benchmarking, an open model/data policy, and Track-1
Combined = F1 + Dice. Consult organizer-specific restrictions before using released reference data.

FFT implementation references:
https://docs.pytorch.org/docs/main/generated/torch.fft.rfft.html
https://docs.pytorch.org/docs/2.13/generated/torch.fft.irfft.html

The automated tests validate code mechanics only. The assistant has not run this notebook on your
Drive-resident noisy/clean corpus or measured a new competition score.